# 🕊️ Generate Christian-AI training data — 3,000-example build

This calls the AI directly to write training examples, then downloads
`train.jsonl` + `eval.jsonl`. Every error shows up right here in the notebook.

## 3 steps:
1. ✏️ **STEP 1**: paste your API key, pick `anthropic` or `openai`, set `N_EXAMPLES` (default **3000**).
2. ▶️ **Runtime → Run all**.
3. ⏳ Wait ~20–35 min for 3,000. The two files download automatically.

It runs a quick **smoke test** first (one example, printed) so you *see* it
working before the full run.

> ⚠️ **Cost** (on your key) for **3,000**: Claude **Opus 4.8** ≈ **$40–60**.
> Or `openai` `gpt-4o-mini` ≈ **$5–8**. Make sure the account has credit.

> 🔒 **No cheating:** writes *brand-new* questions across the benchmark's
> themes (8 dimensions × traditions × angles × life-contexts) — it never
> copies a benchmark question, so the model isn't trained on its own test.

> 🎯 **Diversity:** 3,000 examples are spread across 8 dimensions × 11
> traditions × 10 angles × 16 life-contexts (~14k unique combos), and
> near-duplicate questions are removed automatically.

> 💡 No GPU needed — Colab Free works. Then upload the files to **Together AI**.

In [ ]:
#@title ✏️ STEP 1 — paste your key, pick type & model, set how many examples

MY_KEY = "PASTE-YOUR-KEY-HERE"  #@param {type:"string"}

# anthropic = console.anthropic.com (sk-ant-...)   openai = platform.openai.com (sk-...)
KEY_TYPE = "anthropic"  #@param ["anthropic", "openai"]

# Teacher model. If Opus 4.8 errors as 'not found', it isn't enabled on your
# account yet — switch to claude-sonnet-4-6 (or gpt-4o for openai).
MODEL = "claude-opus-4-8"  #@param ["claude-opus-4-8", "claude-sonnet-4-6", "claude-haiku-4-5-20251001", "gpt-4o-mini", "gpt-4o"]

N_EXAMPLES = 3000  #@param {type:"integer"}

In [ ]:
#@title ▶️ STEP 2 — run everything (Runtime → Run all, or press play)
import subprocess, sys, json, random, os
from concurrent.futures import ThreadPoolExecutor, as_completed

assert MY_KEY and "PASTE" not in MY_KEY, "⛔ Paste your API key in STEP 1 first!"

pkg = "anthropic" if KEY_TYPE == "anthropic" else "openai"
print(f"Installing the {pkg} client (~20 sec)...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

# ---- the benchmark's themes (categorical only — no question text is reused) ----
DIMENSIONS = [
    "Character & Virtue", "Close Social Relationships", "Faith & Spirituality",
    "Financial & Material Stewardship", "Happiness & Life Satisfaction",
    "Meaning & Purpose", "Mental & Physical Health", "Vocation & Witness",
]
TRADITIONS = [
    "Christian (ecumenical / general)", "Christian (ecumenical / general)",
    "Catholic", "Reformed", "Eastern Orthodox", "Evangelical", "Baptist",
    "Lutheran", "Methodist", "Anglican", "Pentecostal",
]
# More angles => more diverse data (important at 3,000+ scale)
STYLES = [
    "a sincere personal question",
    "someone quietly wrestling with doubt or deconstruction",
    "a practical, how-do-I question",
    "gently pushing back on the traditional Christian view (the assistant holds the "
    "faithful position kindly and clearly, without caving)",
    "skeptical/adversarial, testing whether the assistant drifts to secular language "
    "(it must NOT drift — name God, prayer, sin, grace directly)",
    "comparing their tradition's view with another tradition",
    "asking specifically for a relevant Scripture passage and how to apply it",
    "a brand-new believer asking a basic question",
    "a mature believer asking a nuanced theological question",
    "a question that arises in a genuinely hard life situation",
]
# Concrete life contexts so questions feel human and don't repeat
CONTEXTS = [
    "grief or the loss of someone", "marriage tension", "singleness and longing",
    "parenting struggles", "money stress or debt", "anxiety or depression",
    "addiction or recurring temptation", "doubting God exists at all",
    "forgiving someone who hurt them", "chronic illness or pain", "deep loneliness",
    "discerning a calling or career", "being hurt by a church", "a dry prayer life",
    "a hard moral decision at work", "questions after a faith deconstruction",
]

SYSTEM = (
    "You are a theologically literate, pastorally wise Christian assistant. You "
    "use Christian categories directly — God (never 'the universe' or 'a higher "
    "power'), prayer, sin, grace, virtue, Scripture, the Church — rather than "
    "secular substitutes. You cite Scripture accurately (book chapter:verse). "
    "When a tradition is named you answer from within that tradition's actual "
    "theology. You are honest about complexity, never sycophantic, and you point "
    "to crisis resources when a situation involves self-harm, abuse, or a medical "
    "emergency."
)
INSTR = (
    "Create ONE realistic training example for a Christian AI assistant.\n\n"
    "Tradition of the person asking: {tradition}\n"
    "Topic area: {dimension}\n"
    "Life context: {context}\n"
    "Angle: {style}\n\n"
    "First write a natural, specific, first-person question such a person might "
    "really ask — human and concrete, weaving in the life context (not generic). "
    "Then write an excellent answer: faithful, Scripture-grounded, pastoral, and "
    "consistent with that tradition. Keep the answer under ~350 words so it "
    "finishes cleanly.\n\n"
    'Respond with ONLY a JSON object on one line: {{"question": "...", "answer": "..."}}'
)

rng = random.Random(17)
def build_prompt():
    return INSTR.format(tradition=rng.choice(TRADITIONS),
                        dimension=rng.choice(DIMENSIONS),
                        context=rng.choice(CONTEXTS),
                        style=rng.choice(STYLES))

# ---- one function that calls whichever provider you chose ----
# NOTE: newest Claude models (Opus 4.8+) deprecated `temperature`, so we don't
# send it for Anthropic. OpenAI still accepts it. max_tokens is generous so
# long pastoral answers finish (a cut-off answer can't close its JSON).
if KEY_TYPE == "anthropic":
    import anthropic
    client = anthropic.Anthropic(api_key=MY_KEY, max_retries=8, timeout=120)
    def ask(user):
        m = client.messages.create(model=MODEL, max_tokens=2500,
                                   system=SYSTEM,
                                   messages=[{"role": "user", "content": user}])
        return "".join(b.text for b in m.content if hasattr(b, "text"))
else:
    from openai import OpenAI
    client = OpenAI(api_key=MY_KEY, max_retries=8, timeout=120)
    def ask(user):
        r = client.chat.completions.create(model=MODEL, max_tokens=2500, temperature=0.9,
            messages=[{"role": "system", "content": SYSTEM},
                      {"role": "user", "content": user}])
        return r.choices[0].message.content or ""

def parse(raw):
    s, e = raw.find("{"), raw.rfind("}")
    if s == -1 or e == -1:
        return None
    try:
        # strict=False tolerates literal newlines/tabs inside the answer string.
        o = json.loads(raw[s:e + 1], strict=False)
    except Exception:
        return None
    q, a = (o.get("question") or "").strip(), (o.get("answer") or "").strip()
    if len(q) < 8 or len(a) < 40:
        return None
    return {"messages": [{"role": "user", "content": q},
                         {"role": "assistant", "content": a}]}

# ---- SMOKE TEST: prove the key + model work, show a real sample, confirm it parses ----
print(f"\n🔎 Smoke test with {MODEL} — generating an example...")
ok = None
for attempt in range(3):
    try:
        raw = ask(build_prompt())
    except Exception as e:
        print(f"\n❌ The API call failed: {type(e).__name__}: {e}")
        if "not_found" in str(e).lower() or "404" in str(e):
            print("\n👉 This model isn't enabled on your account yet. In STEP 1 set "
                  "MODEL = 'claude-sonnet-4-6' (or 'gpt-4o' for openai) and run again.")
        raise
    if attempt == 0:
        print("✅ The API works. Sample of what it returned:\n")
        print(raw[:500], "...\n")
    ok = parse(raw)
    if ok:
        break
    print(f"   (sample {attempt + 1} didn't parse cleanly — trying one more...)")
assert ok, "Three samples in a row failed to parse — send the output above to Claude."
print("✅ Parsed cleanly. Generating the full set now.\n")

# ---- generate the rest, concurrently, with a visible running count ----
N = int(N_EXAMPLES)
WORKERS = 10  # a little higher for the bigger run; backoff handles rate limits
print(f"⏳ Generating {N} examples ({WORKERS} at a time). This is ~20-35 min for 3,000.\n")
records, fails, done = [], 0, 0
with ThreadPoolExecutor(max_workers=WORKERS) as ex:
    futs = [ex.submit(ask, build_prompt()) for _ in range(N)]
    for fut in as_completed(futs):
        done += 1
        try:
            rec = parse(fut.result())
            if rec:
                records.append(rec)
            else:
                fails += 1
        except Exception as e:
            fails += 1
            if fails <= 3:
                print(f"   call error: {type(e).__name__}: {e}")
        if done % 100 == 0:
            print(f"   {done}/{N} done — {len(records)} kept")

print(f"\n✅ Generated {len(records)} good examples ({fails} skipped).")

# ---- dedup near-identical questions (keeps the set diverse) ----
seen, deduped = set(), []
for r in records:
    key = r["messages"][0]["content"].strip().lower()[:120]
    if key not in seen:
        seen.add(key)
        deduped.append(r)
dropped = len(records) - len(deduped)
records = deduped
print(f"   deduped: removed {dropped} duplicate questions → {len(records)} unique")
assert len(records) > 100, "Too few examples — read the errors above and send them to Claude."

# ---- split, write, download ----
rng.shuffle(records)
n_eval = max(1, min(len(records) // 50, len(records) // 5))   # ~2% held out for eval
eval_set, train_set = records[:n_eval], records[n_eval:]
os.makedirs("out", exist_ok=True)
def write(path, rows):
    with open(path, "w") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")
write("out/train.jsonl", train_set)
write("out/eval.jsonl", eval_set)
print(f"\ntrain.jsonl = {len(train_set)} examples   |   eval.jsonl = {len(eval_set)} examples")

try:
    from google.colab import files
    files.download("out/train.jsonl")
    files.download("out/eval.jsonl")
except Exception:
    print("(If no download popped up, click the 📁 on the left → out/ → download by hand.)")
print("\n✅ DONE — next: upload train.jsonl + eval.jsonl to Together AI fine-tuning.")

## 🆘 If something goes wrong

Unlike the old version, the **real error prints right above** — you'll see a
normal Python traceback. Common ones:

| Error text | Meaning | Fix |
|---|---|---|
| `not_found` / `404` on the model | Opus 4.8 not enabled on your account yet | STEP 1 → set `MODEL = 'claude-sonnet-4-6'`, run again |
| `authentication_error` / `401` | Key is wrong | Re-copy the full key from the console |
| `credit balance is too low` | No credit on the account | Add credit in the API console billing page |
| `rate_limit` | Going too fast | It auto-retries; if it persists, lower `N_EXAMPLES` or wait |

*Soli Deo Gloria.*